# RLHF Fine-Tuning

## Load the SFT and Reward Models

In [1]:
# %cp /content/drive/MyDrive/copy\ files/reward_model.pt .
# %cp /content/drive/MyDrive/copy\ files/sft_model_epoch_1.zip .

In [2]:
# !unzip sft_model_epoch_1.zip

## Reward model

In [3]:
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class RewardHead(nn.Module):
    """
    The RewardHead class implements a head for GPT2
    that returns a scalar for each output token.
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(self.reward.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.reward(output)


class GPT2RewardModel(nn.Module):
    """
    GPT2 model with a reward head on top.
    """

    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        # config = self.llm.config
        # Add the reward head
        self.reward_head = RewardHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:

        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )

        # Get the last hidden state
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # Apply the reward head
        rewards = self.reward_head(last_hidden_state).squeeze(-1)

        return torch.sigmoid(rewards)



/home/talos-1/miniforge3/envs/torch_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_name = "gpt2"
reward_model = GPT2RewardModel(model_name)
reward_model.load_state_dict(torch.load("reward_model.pt", map_location='cpu'))

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2031.72it/s]


<All keys matched successfully>

## Model with Value Head

In [5]:
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class ValueHead(nn.Module):
    """
    The ValueHead class implements a head for GPT2
    that returns a scalar for each output token.
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.value = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(self.value.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.value.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.value(output)


class ModelForCausalLMWithValueHead(nn.Module):
    """
    GPT2 model with a value head on top.
    """

    def __init__(self, model_path):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_path)
        # config = self.llm.config
        # Add the reward head
        self.v_head = ValueHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:

        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )
        lm_logits = transformer_outputs.logits
        # Get the last hidden state
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # Apply the reward head
        value = self.v_head(last_hidden_state).squeeze(-1)
        return lm_logits, value

    def generate(self, *args, **kwargs):
        return self.llm.generate(*args, **kwargs)


In [6]:
model_path = './sft_model_epoch_1'
model = ModelForCausalLMWithValueHead(model_path)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1916.74it/s]


## Preparing Dataset

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [8]:
# %pip install datasets==3.5.0

In [9]:
from datasets import load_dataset
dataset = load_dataset("sst2")
dataset

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [10]:
ds_train, ds_val = dataset['train'], dataset['validation']

## Filtering

In [11]:
len(ds_train)

67349

In [12]:
ds_train = ds_train.filter(lambda x: len(x['sentence'].split(' ')) > 8)

In [13]:
len(ds_train)

31105

In [14]:
ds_val = ds_val.filter(lambda x: len(x['sentence'].split(' ')) > 8)

In [15]:
len(ds_val)

807

In [16]:
import random
input_min_token_length = 2
input_max_token_length = 8
input_token_length_range = list(range(input_min_token_length, input_max_token_length))
print(input_token_length_range)

[2, 3, 4, 5, 6, 7]


In [17]:
random.choice(input_token_length_range)

2

In [18]:
def tokenize(sample):
    input_size = random.choice(input_token_length_range)
    sample['input_ids'] = tokenizer.encode(sample['sentence'])[:input_size]
    sample['attention_mask'] = [1] * len(sample['input_ids'])
    sample['query'] = tokenizer.decode(sample['input_ids'])
    return sample

map_kwargs = {
    "batched": False,
    "remove_columns": ['idx', 'sentence', 'label']
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)


In [19]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [20]:
tokenized_dataset_train[6]

{'input_ids': tensor([ 1640,   883,  3807, 31006,   508, 13121,   326]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1]),
 'query': 'for those moviegoers who complain that'}

In [21]:
REWARD_TOKEN_ID = tokenizer.eos_token_id

In [22]:
from torch.utils.data import DataLoader

batch_size = 32

def collator(batch):
    return dict((key, [d[key] for d in batch]) for key in batch[0])

train_dataloader = DataLoader(tokenized_dataset_train, batch_size=batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(tokenized_dataset_val, batch_size=batch_size, collate_fn=collator, shuffle=True)

In [23]:
batch = next(iter(train_dataloader))
batch

{'input_ids': [tensor([270, 705,  82, 407]),
  tensor([43764,   640,   837,   326,   705,    82,   477]),
  tensor([16670,  6547]),
  tensor([   79,  2350,   534,  1393,   837,   534, 13843]),
  tensor([270, 705]),
  tensor([271, 299, 488]),
  tensor([   64,  4159,    12, 19316, 11783, 28763,   286]),
  tensor([8807,  262,  749, 1327]),
  tensor([   11, 17605,   261,   468,   284, 46528, 34377]),
  tensor([ 568,  881,  546,  262, 2646]),
  tensor([   64,  5442, 10997,   351,   663]),
  tensor([42552,   341]),
  tensor([ 82, 676]),
  tensor([  271, 24097,   837,  4591,  1536]),
  tensor([1462, 7264,  837]),
  tensor([  64, 6036,   88]),
  tensor([ 1326, 34006]),
  tensor([ 301, 1886, 3685]),
  tensor([  361,   484,  6265,   503,   656, 15962, 30569]),
  tensor([11722, 21557,  1636,   318,  4988,  8258]),
  tensor([  732,   705,   303,  8288,   479, 33663]),
  tensor([  86,  346, 1073, 3296,  481]),
  tensor([ 5171,   407,  1037,   307, 34988,   416]),
  tensor([   64, 21247,   284,   26

In [24]:
output_min_length = 5
output_max_length = 16

# https://huggingface.co/docs/trl/how_to_train#how-to-generate-text-for-training

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id
}

## Sample Generation

In [25]:
new_tokens = random.choice(list(range(output_min_length, output_max_length)))
generation_kwargs["max_new_tokens"] = new_tokens
sample = tokenizer('Hi, this')
sample

{'input_ids': [17250, 11, 428], 'attention_mask': [1, 1, 1]}

In [26]:
query_response = model.generate(
    input_ids=torch.tensor(sample['input_ids']).unsqueeze(0),
    attention_mask=torch.tensor(sample['attention_mask']).unsqueeze(0),
    **generation_kwargs
    ).squeeze(0)
query_response

tensor([17250,    11,   428,   318,  4713,   329,   262, 11278,  3159,   220,
          666,  8461,  8781,  9155,   319,   329,   257])

In [27]:
tokenizer.decode(query_response)

'Hi, this is fresh for the explosion screen ianni lets loose on for a'

In [28]:
with torch.no_grad():
    query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID])])
    attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
    score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
score

tensor(0.9634)

## Batch Generation

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
reward_model = reward_model.to(device)

query_tensors = batch['input_ids']
query_attention_masks = batch['attention_mask']

response_tensors = []
query_response_tensors = []
score_tensors = []

for i, query in enumerate(query_tensors):
    query = query.to(device)
    query_attention_mask = query_attention_masks[i].to(device)
    new_tokens = random.choice(list(range(output_min_length, output_max_length)))
    generation_kwargs["max_new_tokens"] = new_tokens
    query_response = model.generate(
        input_ids=query.unsqueeze(0),
        attention_mask=query_attention_mask.unsqueeze(0),
        **generation_kwargs
    ).squeeze(0)

    response_len = len(query_response) - len(query)
    response_tensors.append(query_response[-response_len:])
    query_response_tensors.append(query_response)

    with torch.no_grad():
        query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])
        attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
        score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
        score = 2 * (score - 0.5)
    score_tensors.append(score)

batch["response"] = [tokenizer.decode(response) for response in response_tensors]
print(batch['response'])

[" very good ... \xa0 it 's not terribly interesting , and", " it 's about .        ", ' shallow ian title tale  ian', ' and your empathy          ', 's dark-hearted , just iced iced iced iced', "olson 's none-of-the-", ' the famous animated movie .', '-fought triumph .      ', ' stereotypes like pervert hollywood dealer', ' -- misguided ruminations on the', ' two gifted actresses ive harras ia as', " comes from a script that 's terribly underused , hampered by out", ' into our own black comedy flicks and', ' , and icky iced iced it in iced tea iced', ' even after 15 minutes ,', 'bopper mall type .   ', ' my paycheck and my time', '      indulged .     ', 'ography midway in the film , why bother', ' from start to finish .        ', ' since its first incarnation ive seen them in a romantic context', ' probably find this translation frighteningly rewarding .      ', ' how all of four leads are the same', ' sad songs of our times       ', ' mugging tactics and the', ' subversive new trans

## Compute Reward

$\text {reward} = \text {score} - \log (\frac {\pi^{RL}_\theta} {\pi^{SFT}})$

In [30]:
from copy import deepcopy
sft_model = deepcopy(model)

In [31]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [32]:
input_data = data_collator([
    {'input_ids': ids,
     'attention_mask': torch.ones_like(ids)} for ids in query_response_tensors
]).to(device)
input_data

{'input_ids': tensor([[  270,   705,    82,   407,   845,   922,  2644,   220,  1849,   340,
           705,    82,   407, 22121,  3499,   837,   290, 50256, 50256, 50256],
        [43764,   640,   837,   326,   705,    82,   477,   340,   705,    82,
           546,   764,   220,   220,   220,   220,   220,   220,   220,   220],
        [16670,  6547, 19337,   220,   666,  3670, 12838,   220,   220,   666,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
        [   79,  2350,   534,  1393,   837,   534, 13843,   290,   534, 21452,
           220,   220,   220,   220,   220,   220,   220,   220,   220,   220],
        [  270,   705,    82,  3223,    12, 20122,   837,   655,   220,  3711,
           220,  3711,   220,  3711,   220,  3711, 50256, 50256, 50256, 50256],
        [  271,   299,   488, 32836,   705,    82,  4844,    12,  1659,    12,
          1169,    12, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
        [   64,  4159,    12, 19

In [33]:
def compute_rewards(input_data, query_tensors, response_tensors, score_tensors):
    with torch.no_grad():
        logits, values = model(**input_data) # b, seq, vocab
        ref_logits, _ = sft_model(**input_data)
        logp = torch.nn.functional.log_softmax(logits[:, :-1, :], dim=-1)
        ref_logp = torch.nn.functional.log_softmax(ref_logits[:, :-1, :], dim=-1)

        labels = input_data['input_ids'][:, 1:] # b, seq

        logp = torch.gather(logp, 2, labels.unsqueeze(-1)).squeeze(-1) # batch, seq
        ref_logp = torch.gather(ref_logp, 2, labels.unsqueeze(-1)).squeeze(-1) # batch, seq

        kl = logp - ref_logp
        beta = 0.2
        rewards = - beta * kl
        attention_mask = input_data['attention_mask']
        masks = torch.zeros_like(attention_mask[:, 1:])
        masks[:,:] = attention_mask[:, 1:]
        for j in range(len(query_tensors)):
            start = len(query_tensors[j]) - 1
            end = start + len(response_tensors[j])
            masks[j, :start] = 0
            masks[j, end:] = 0
            rewards[j, end - 1] += score_tensors[j]
            rewards[j, :] *= masks[j, :]
            values[j, :-1] *= masks[j, :]

    return logp, rewards, values[:, :-1], masks


In [34]:
logprobs, rewards, values, masks = compute_rewards(input_data, query_tensors, response_tensors, score_tensors)
print(rewards[0])
print(input_data['input_ids'][0])
print(input_data['attention_mask'][0])

tensor([-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
        -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.9934,
        -0.0000, -0.0000, -0.0000], device='cuda:0')
tensor([  270,   705,    82,   407,   845,   922,  2644,   220,  1849,   340,
          705,    82,   407, 22121,  3499,   837,   290, 50256, 50256, 50256],
       device='cuda:0')
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
       device='cuda:0')


In [35]:
print(masks[0])
print(values[0])

tensor([0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
       device='cuda:0')
tensor([-0.0000, -0.0000, -0.0000, -1.7374, -2.3729, -1.8228, -1.5497, -1.2572,
        -2.0840, -2.3472, -1.0304, -1.7560, -2.4944, -2.6816, -1.9973, -2.1710,
        -0.0000, -0.0000, -0.0000], device='cuda:0')


## Compute Advantage

In [36]:
def masked_mean(values, mask):
    return (values * mask).sum() / mask.sum()

def masked_var(values, mask):
    mean = masked_mean(values, mask)
    centred_values = values - mean
    return masked_mean(centred_values ** 2, mask)

def masked_whiten(values, mask):
    mean, var = masked_mean(values, mask), masked_var(values, mask)
    whitened = (values - mean) * torch.rsqrt(var + 1e-8)
    whitened += mean
    return whitened

def compute_advantage(rewards, values, masks):
    lastgae = 0.0
    advantage_reversed = []
    seq_length = rewards.shape[-1]
    gamma, lam = 1.0, 0.95

    for t in reversed(range(seq_length)):
        nextvalues = values[:, t + 1] if t < seq_length - 1 else 0.0
        delta = rewards[:, t] + gamma * nextvalues - values[:, t]
        lastgae = delta + gamma * lam * lastgae
        advantage_reversed.append(lastgae)
    advantages = torch.stack(advantage_reversed[::-1], dim=1)
    advantages = masked_whiten(advantages, masks)

    returns = advantages + values
    return advantages, returns


In [37]:
advantages, returns = compute_advantage(rewards, values, masks)
print(advantages[0])
print(returns[0])

tensor([-0.7889, -0.8448, -0.9037,  0.5249,  1.0833,  0.6540,  0.4397,  0.1974,
         0.9027,  1.1616,  0.0786,  0.6909,  1.3464,  1.5634,  1.0441,  1.2337,
         0.2739,  0.2739,  0.2739], device='cuda:0')
tensor([-0.7889, -0.8448, -0.9037, -1.2125, -1.2896, -1.1688, -1.1100, -1.0598,
        -1.1812, -1.1855, -0.9517, -1.0651, -1.1481, -1.1182, -0.9531, -0.9373,
         0.2739,  0.2739,  0.2739], device='cuda:0')


## Mini-batch PPO Training

### Training Config

In [38]:
learning_rate = 1e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [39]:
np.random.permutation(batch_size)

array([20,  7,  1, 28, 30,  5, 26, 29,  9, 22, 19, 24,  3, 18,  4, 16, 31,
       21, 25, 11, 13, 15, 27,  8, 10, 14,  6, 12, 17,  0, 23,  2])

In [40]:
mini_batch_size = 4
ppo_epochs = 4

cliprange_ratio = 0.2

v_loss_coeff = 0.1

ratio_threshold = 10

def compute_loss(old_logprobs, values, logprobs, vpreds, masks, advantages, returns):
    ratio = torch.exp(logprobs - old_logprobs)
    pg_loss1 = - ratio * advantages
    pg_loss2 = - torch.clamp(ratio, 1 - cliprange_ratio, 1 + cliprange_ratio) * advantages
    pg_loss = masked_mean(torch.max(pg_loss1, pg_loss2), masks)

    v_loss = masked_mean((vpreds - returns) ** 2, masks)
    loss = pg_loss + v_loss_coeff * v_loss

    avg_ratio = masked_mean(ratio, masks)
    if avg_ratio > ratio_threshold:
        pg_loss = pg_loss * 0.0
        v_loss = v_loss * 0.0
        loss = loss * 0.0

    return loss, v_loss

def mini_batch_train():
    current_batch_size = input_data['input_ids'].size(0)

    for ep in range(ppo_epochs):
        batch_inds = np.random.permutation(current_batch_size)

        for start in range(0, current_batch_size, mini_batch_size):
            end = start + mini_batch_size
            mini_batch_inds = batch_inds[start:end]

            mb_model_inputs = {
                'input_ids': input_data['input_ids'][mini_batch_inds],
                'attention_mask': input_data['attention_mask'][mini_batch_inds]
            }
            mb_logits, mb_vpreds = model(**mb_model_inputs)
            mb_logits = torch.nn.functional.log_softmax(mb_logits[:, :-1, :], dim=-1)
            mb_logprobs = torch.gather(mb_logits, 2, mb_model_inputs['input_ids'][:, 1:].unsqueeze(-1)).squeeze(-1)

            loss, loss_v = compute_loss(
                logprobs[mini_batch_inds],
                values[mini_batch_inds],
                mb_logprobs,
                mb_vpreds[:, :-1],
                masks[mini_batch_inds],
                advantages[mini_batch_inds],
                returns[mini_batch_inds]
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            print('loss/total', loss.item())
    print('mini-batch training finished')



In [41]:
mini_batch_train()

loss/total -1.2503538131713867
loss/total -1.5623435974121094
loss/total -1.73579740524292
loss/total -1.260434627532959
loss/total -0.9313328862190247
loss/total -1.0947372913360596
loss/total -0.9795888662338257
loss/total -0.9816614389419556
loss/total -1.8145575523376465
loss/total -1.4270374774932861
loss/total -1.699352502822876
loss/total -1.1914849281311035
loss/total -1.2174855470657349
loss/total -1.6139088869094849
loss/total -1.3183300495147705
loss/total -1.6611032485961914
loss/total -1.5740468502044678
loss/total -1.5588781833648682
loss/total -1.4295694828033447
loss/total -1.480506181716919
loss/total -1.65351402759552
loss/total -1.2841209173202515
loss/total -2.0284698009490967
loss/total -1.2624623775482178
loss/total -1.3395603895187378
loss/total -1.366981029510498
loss/total -1.6159449815750122
loss/total -0.9589351415634155
loss/total -2.297910213470459
loss/total -1.418696403503418
loss/total -1.7240042686462402
loss/total -2.35037899017334
mini-batch training 

## Train RLHF

In [42]:
num_epochs = 1

for epoch in range(num_epochs):
    for batch in train_dataloader:
        # Generate responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']

        response_tensors = []
        query_response_tensors = []
        score_tensors = []

        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = random.choice(list(range(output_min_length, output_max_length)))
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
                ).squeeze(0)

            response_len = len(query_response) - len(query)
            response_tensors.append(query_response[-response_len:])
            query_response_tensors.append(query_response)

            with torch.no_grad():
                query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])
                attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
                score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
                score = 2 * (score - 0.5)
            score_tensors.append(score)

        input_data = data_collator([
            {
                'input_ids': ids,
                'attention_mask': torch.ones_like(ids)
            }
            for ids in query_response_tensors
        ]).to(device)

        # rewards and advantages
        logprobs, rewards, values, masks = compute_rewards(input_data, query_tensors, response_tensors, score_tensors)
        advantages, returns = compute_advantage(rewards, values, masks)

        # mini batch training
        mini_batch_train()
    print(f'epoch {epoch + 1} finished')

loss/total -0.10720986127853394
loss/total 0.2065025269985199
loss/total -0.45473939180374146
loss/total 0.3123462498188019
loss/total -0.46576327085494995
loss/total 0.47024625539779663
loss/total 0.11858776211738586
loss/total -0.00014282390475273132
loss/total -0.8373959064483643
loss/total 0.02571558952331543
loss/total 0.3584193289279938
loss/total 0.31940820813179016
loss/total 0.13551348447799683
loss/total -0.4388121962547302
loss/total -0.22875913977622986
loss/total 0.20411187410354614
loss/total -0.10027123987674713
loss/total 0.3153979480266571
loss/total 0.053266704082489014
loss/total -0.3576067090034485
loss/total 0.06324337422847748
loss/total -0.4241448938846588
loss/total -0.5178835391998291
loss/total 0.15040050446987152
loss/total -0.0495033897459507
loss/total 0.20507948100566864
loss/total 0.05915984883904457
loss/total -0.8973820209503174
loss/total -0.39511746168136597
loss/total -0.15620802342891693
loss/total -0.15623262524604797
loss/total 0.23815099895000458

## Validation

In [43]:
len(tokenized_dataset_val)

807

In [44]:
val_gen_lengths = [0] * len(tokenized_dataset_val)
for i in range(len(tokenized_dataset_val)):
    val_gen_lengths[i] = random.choice(list(range(output_min_length, output_max_length)))

In [45]:
val_gen_lengths[:10]

[5, 10, 13, 14, 12, 8, 7, 6, 11, 15]

In [46]:
def validate():
    scores = []
    for b, batch in enumerate(val_dataloader):
        # Generate_responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']
        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = val_gen_lengths[b * len(query_tensors) + i]
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
                ).squeeze(0)
            query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])
            attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
            score = reward_model(query_response_score.unsqueeze(0), attention_mask.unsqueeze(0)).squeeze(0)[-1]
            score = 2 * (score - 0.5)
            scores.append(score.item())
    print('avg score:', sum(scores) / len(scores))

In [47]:
validate()

avg score: 0.7369496158359957


In [48]:
torch.save(model.state_dict(), 'ppo_model_epoch_1.pt')

In [49]:
model_path = './sft_model_epoch_1'
model = ModelForCausalLMWithValueHead(model_path).to(device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1751.42it/s]


In [50]:
validate()

avg score: 0.10329635886306952
